In [ ]:
import pandas as pd
import os
import glob
import numpy as np

def convert_csv_format(input_folder, output_file):
    """
    将多个CSV文件从原始格式转换为目标格式并合并
    
    参数:
        input_folder: 输入CSV文件所在的文件夹路径
        output_file: 输出CSV文件的路径
    """
    # 获取所有CSV文件
    csv_files = glob.glob(os.path.join(input_folder, "*.csv"))
    
    # 初始化一个空的数据框来存储所有数据
    all_data = pd.DataFrame()
    
    # 处理每个CSV文件
    for file_path in csv_files:
        try:
            # 读取CSV文件
            df = pd.read_csv(file_path, sep=',',header=None, names=['ch', 'phase_index', 'target', 'mag', 'phase'])
            S_df = df[df['phase']=='S']
            S_df.rename(columns={'phase_index':'s_target'},inplace = True)

            P_df = df[df['phase']=='P']
            P_df.rename(columns={'phase_index':'p_target'},inplace = True)
            merge_df = pd.merge(S_df, P_df, how='inner' , on=['ch'])
           

            
            # 创建目标格式的数据框
            converted_df = pd.DataFrame()
            
            # 转换格式
            # Key: 可能需要根据Key1和Key2组合而成，这里假设是Key1_Key2
            converted_df['Key'] = file_path.split("/")[-1][:-4] + '_' + merge_df['ch'].astype(str)
            
            # p_target和s_target: 从target和phase计算得到
            # 假设target是幅度，phase是相位，那么:
            # p_target = target * cos(phase)
            # s_target = target * sin(phase)
            converted_df['p_target'] = merge_df['p_target']
            converted_df['s_target'] = merge_df['s_target']
            
            # Dis: 可能需要计算或保留为空
            converted_df['Dis'] = np.nan  # 暂时设为NaN
            
            # Mag_value: 直接使用mag列
            converted_df['Mag_value'] = merge_df['mag_x']
            
            # From: 记录来源文件名
            converted_df['From'] = "xfj3km"
            
            # snr: 可能需要计算或保留为空
            converted_df['snr'] = np.nan  # 暂时设为NaN
            
            # datasplit: 可能需要根据某些规则分配
            converted_df['datasplit'] = 'train'  # 默认设为train
            
            # shot: 可能需要从文件名或其他信息提取
            converted_df['shot'] = 100  # 默认设为1
            
            # 添加到总数据框
            all_data = pd.concat([all_data, converted_df], ignore_index=True)
            print(converted_df)
            print(f"已处理文件: {file_path}")
            
        except Exception as e:
            print(f"处理文件 {file_path} 时出错: {str(e)}")
    
    # 保存合并后的数据
    if not all_data.empty:
        all_data.loc[2700:, "datasplit"] = "val"
        all_data.to_csv(output_file, index=False)
        print(f"已保存合并后的数据到: {output_file}")
        print(f"总记录数: {len(all_data)}")
    else:
        print("没有找到任何有效数据")

# 使用示例
if __name__ == "__main__":
    input_folder = "/home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/phase_picks"  # 替换为您的CSV文件所在文件夹路径
    output_file = "/home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/phase_picks/data/merged_data_1000.csv"  # 替换为您想要的输出文件路径
    
    convert_csv_format(input_folder, output_file)

                       Key  p_target  s_target  Dis  Mag_value    From  snr  \
0    xfj_das_100Hz_149_290      2292      2426  NaN       -0.2  xfj3km  NaN   
1    xfj_das_100Hz_149_291      2292      2426  NaN       -0.2  xfj3km  NaN   
2    xfj_das_100Hz_149_292      2292      2426  NaN       -0.2  xfj3km  NaN   
3    xfj_das_100Hz_149_293      2291      2426  NaN       -0.2  xfj3km  NaN   
4    xfj_das_100Hz_149_294      2291      2426  NaN       -0.2  xfj3km  NaN   
..                     ...       ...       ...  ...        ...     ...  ...   
673  xfj_das_100Hz_149_906      2287      2422  NaN       -0.2  xfj3km  NaN   
674  xfj_das_100Hz_149_907      2287      2422  NaN       -0.2  xfj3km  NaN   
675  xfj_das_100Hz_149_907      2287      2422  NaN       -0.2  xfj3km  NaN   
676  xfj_das_100Hz_149_908      2287      2422  NaN       -0.2  xfj3km  NaN   
677  xfj_das_100Hz_149_908      2287      2422  NaN       -0.2  xfj3km  NaN   

    datasplit  shot  
0       train   100  
1      

/tmp/ipykernel_2412528/2199358944.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  S_df.rename(columns={'phase_index':'s_target'},inplace = True)
/tmp/ipykernel_2412528/2199358944.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  P_df.rename(columns={'phase_index':'p_target'},inplace = True)
/tmp/ipykernel_2412528/2199358944.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  S_df.rename(columns={'phase_index':'s_target'},inpla

In [11]:
import numpy as np
import h5py
import os
import glob
import argparse
import yaml
from tqdm import tqdm

def convert_npy_format(input_folder, output_file):
    """
    将多个CSV文件从原始格式转换为目标格式并合并
    
    参数:
        input_folder: 输入CSV文件所在的文件夹路径
        output_file: 输出CSV文件的路径
    """
    # 获取所有CSV文件
    csv_files = glob.glob(os.path.join(input_folder, "*.csv"))
    
    # 初始化一个空的数据框来存储所有数据
    all_data = pd.DataFrame()
    
    # 处理每个CSV文件
    with h5py.File(output_file, 'w') as h5f:
        for file_path in csv_files:
            
            # 读取CSV文件
            df = pd.read_csv(file_path, sep=',',header=None, names=['ch', 'phase_index', 'target', 'mag', 'phase'])
            S_df = df[df['phase']=='S']
            S_df.rename(columns={'target':'s_target'},inplace = True)

            P_df = df[df['phase']=='P']
            P_df.rename(columns={'target':'p_target'},inplace = True)
            merge_df = pd.merge(S_df, P_df, how='inner' , on=['ch'])


            npy_file_path = "/home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/data/"+file_path.split("/")[-1][:-4]+".npy"
            
            # 加载NPY文件
            data = np.load(npy_file_path)
            # 检查数据维度
            if len(data.shape) != 2:
                print(f"跳过文件 {file_path}: 数据不是二维数组 (形状: {data.shape})")
                continue
            
            # 获取文件名（不含扩展名）作为数据集名称
            file_name = os.path.splitext(os.path.basename(file_path))[0]
            
            # 选择特定列
            columns =list(set(merge_df['ch'])) 
            for ch in columns:
                selected_data = np.array([data[ch],data[ch],data[ch]]).T
                print(selected_data.shape)
                dataset_name = file_name + '_' + str(ch)
                print(dataset_name)
                # 将数据保存到H5文件
                try:
                    h5f.create_dataset(dataset_name, data=selected_data)
                
                except Exception as e:
                    print(f"writing {dataset_name} 时出错: {str(e)}")
            
        # 添加全局属性
    
    print(f"数据已成功保存到 {output_file}")
convert_npy_format( "/home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/phase_picks" , "/home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/phase_picks/data/merged_data_1000.h5")

/tmp/ipykernel_2412528/3235321997.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  S_df.rename(columns={'target':'s_target'},inplace = True)
/tmp/ipykernel_2412528/3235321997.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  P_df.rename(columns={'target':'p_target'},inplace = True)


(8000, 3)
xfj_das_100Hz_149_290
(8000, 3)
xfj_das_100Hz_149_291
(8000, 3)
xfj_das_100Hz_149_292
(8000, 3)
xfj_das_100Hz_149_293
(8000, 3)
xfj_das_100Hz_149_294
(8000, 3)
xfj_das_100Hz_149_295
(8000, 3)
xfj_das_100Hz_149_296
(8000, 3)
xfj_das_100Hz_149_297
(8000, 3)
xfj_das_100Hz_149_298
(8000, 3)
xfj_das_100Hz_149_299
(8000, 3)
xfj_das_100Hz_149_300
(8000, 3)
xfj_das_100Hz_149_301
(8000, 3)
xfj_das_100Hz_149_302
(8000, 3)
xfj_das_100Hz_149_303
(8000, 3)
xfj_das_100Hz_149_304
(8000, 3)
xfj_das_100Hz_149_305
(8000, 3)
xfj_das_100Hz_149_306
(8000, 3)
xfj_das_100Hz_149_307
(8000, 3)
xfj_das_100Hz_149_308
(8000, 3)
xfj_das_100Hz_149_309
(8000, 3)
xfj_das_100Hz_149_310
(8000, 3)
xfj_das_100Hz_149_311
(8000, 3)
xfj_das_100Hz_149_312
(8000, 3)
xfj_das_100Hz_149_313
(8000, 3)
xfj_das_100Hz_149_314
(8000, 3)
xfj_das_100Hz_149_315
(8000, 3)
xfj_das_100Hz_149_316
(8000, 3)
xfj_das_100Hz_149_317
(8000, 3)
xfj_das_100Hz_149_318
(8000, 3)
xfj_das_100Hz_149_319
(8000, 3)
xfj_das_100Hz_149_320
(8000, 3

/tmp/ipykernel_2412528/3235321997.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  S_df.rename(columns={'target':'s_target'},inplace = True)
/tmp/ipykernel_2412528/3235321997.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  P_df.rename(columns={'target':'p_target'},inplace = True)


(8000, 3)
xfj_das_100Hz_150_897
(8000, 3)
xfj_das_100Hz_150_898
(8000, 3)
xfj_das_100Hz_150_899
(8000, 3)
xfj_das_100Hz_150_900
(8000, 3)
xfj_das_100Hz_150_901
(8000, 3)
xfj_das_100Hz_150_902
(8000, 3)
xfj_das_100Hz_150_903
(8000, 3)
xfj_das_100Hz_150_904
(8000, 3)
xfj_das_100Hz_150_905
(8000, 3)
xfj_das_100Hz_150_906
(8000, 3)
xfj_das_100Hz_150_907
(8000, 3)
xfj_das_100Hz_150_908
(8000, 3)
xfj_das_100Hz_150_909
(8000, 3)
xfj_das_100Hz_150_910
(8000, 3)
xfj_das_100Hz_150_911
(8000, 3)
xfj_das_100Hz_150_912
(8000, 3)
xfj_das_100Hz_150_913
(8000, 3)
xfj_das_100Hz_150_914
(8000, 3)
xfj_das_100Hz_150_915
(8000, 3)
xfj_das_100Hz_150_916
(8000, 3)
xfj_das_100Hz_150_917
(8000, 3)
xfj_das_100Hz_150_918
(8000, 3)
xfj_das_100Hz_150_919
(8000, 3)
xfj_das_100Hz_150_920
(8000, 3)
xfj_das_100Hz_150_921
(8000, 3)
xfj_das_100Hz_150_922
(8000, 3)
xfj_das_100Hz_150_923
(8000, 3)
xfj_das_100Hz_150_924
(8000, 3)
xfj_das_100Hz_150_925
(8000, 3)
xfj_das_100Hz_150_926
(8000, 3)
xfj_das_100Hz_150_927
(8000, 3

/tmp/ipykernel_2412528/3235321997.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  S_df.rename(columns={'target':'s_target'},inplace = True)
/tmp/ipykernel_2412528/3235321997.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  P_df.rename(columns={'target':'p_target'},inplace = True)
/tmp/ipykernel_2412528/3235321997.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  S_df.rename(columns={'target':'s_target'},inplace = True)
/tmp

...

(8000, 3)
xfj_das_100Hz_405_238
(8000, 3)
xfj_das_100Hz_405_239
(8000, 3)
xfj_das_100Hz_405_240
(8000, 3)
xfj_das_100Hz_405_241
(8000, 3)
xfj_das_100Hz_405_242
(8000, 3)
xfj_das_100Hz_405_243
(8000, 3)
xfj_das_100Hz_405_244
(8000, 3)
xfj_das_100Hz_405_245
(8000, 3)
xfj_das_100Hz_405_246
(8000, 3)
xfj_das_100Hz_405_247
(8000, 3)
xfj_das_100Hz_405_248
(8000, 3)
xfj_das_100Hz_405_249
(8000, 3)
xfj_das_100Hz_405_250
(8000, 3)
xfj_das_100Hz_405_251
(8000, 3)
xfj_das_100Hz_405_252
(8000, 3)
xfj_das_100Hz_405_253
(8000, 3)
xfj_das_100Hz_405_254
(8000, 3)
xfj_das_100Hz_405_255
(8000, 3)
xfj_das_100Hz_405_256
(8000, 3)
xfj_das_100Hz_405_257
(8000, 3)
xfj_das_100Hz_405_258
(8000, 3)
xfj_das_100Hz_405_259
(8000, 3)
xfj_das_100Hz_405_260
(8000, 3)
xfj_das_100Hz_405_261
(8000, 3)
xfj_das_100Hz_405_262
(8000, 3)
xfj_das_100Hz_405_263
(8000, 3)
xfj_das_100Hz_405_264
(8000, 3)
xfj_das_100Hz_405_265
(8000, 3)
xfj_das_100Hz_405_266
(8000, 3)
xfj_das_100Hz_405_267
(8000, 3)
xfj_das_100Hz_405_268
(8000, 3

/tmp/ipykernel_2412528/3235321997.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  S_df.rename(columns={'target':'s_target'},inplace = True)
/tmp/ipykernel_2412528/3235321997.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  P_df.rename(columns={'target':'p_target'},inplace = True)


(8000, 3)
xfj_das_100Hz_406_297
(8000, 3)
xfj_das_100Hz_406_298
(8000, 3)
xfj_das_100Hz_406_299
(8000, 3)
xfj_das_100Hz_406_300
(8000, 3)
xfj_das_100Hz_406_301
(8000, 3)
xfj_das_100Hz_406_302
(8000, 3)
xfj_das_100Hz_406_303
(8000, 3)
xfj_das_100Hz_406_304
(8000, 3)
xfj_das_100Hz_406_305
(8000, 3)
xfj_das_100Hz_406_306
(8000, 3)
xfj_das_100Hz_406_307
(8000, 3)
xfj_das_100Hz_406_308
(8000, 3)
xfj_das_100Hz_406_309
(8000, 3)
xfj_das_100Hz_406_310
(8000, 3)
xfj_das_100Hz_406_311
(8000, 3)
xfj_das_100Hz_406_312
(8000, 3)
xfj_das_100Hz_406_313
(8000, 3)
xfj_das_100Hz_406_314
(8000, 3)
xfj_das_100Hz_406_315
(8000, 3)
xfj_das_100Hz_406_316
(8000, 3)
xfj_das_100Hz_406_317
(8000, 3)
xfj_das_100Hz_406_318
(8000, 3)
xfj_das_100Hz_406_319
(8000, 3)
xfj_das_100Hz_406_320
(8000, 3)
xfj_das_100Hz_406_321
(8000, 3)
xfj_das_100Hz_406_322
(8000, 3)
xfj_das_100Hz_406_323
(8000, 3)
xfj_das_100Hz_406_324
(8000, 3)
xfj_das_100Hz_406_325
(8000, 3)
xfj_das_100Hz_406_326
(8000, 3)
xfj_das_100Hz_406_327
(8000, 3

/tmp/ipykernel_2412528/3235321997.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  P_df.rename(columns={'target':'p_target'},inplace = True)
/tmp/ipykernel_2412528/3235321997.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  P_df.rename(columns={'target':'p_target'},inplace = True)


FileNotFoundError: [Errno 2] No such file or directory: '/home/disk/disk02/wzm/DAS_DL_Dataset/DASEventData/data/xfj_das_100Hz_408.npy'

In [7]:

import h5py
import numpy as np
with h5py.File("/home/disk/disk02/wzm/Sustech_Pulse/src/finetune/wzm/data_new/course_data_dpk.h5", 'r') as f:
    data = np.array(f.get("29416.0629"))
    print(data)

[[555. 271.  59.]
 [521. 350. 114.]
 [576. 336.  71.]
 ...
 [699. 283. -36.]
 [591. 346.   6.]
 [550. 421.  76.]]
